# [Chapter 16 — Spatially and Continuously Structured Models, §16.2] Age-Structured Population: The von Foerster Equation

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 16 — Spatially and Continuously Structured Models
**Considerations developed:** 5 (linearization fidelity), 7 (mixing balance in continuous structure)
**Estimated runtime:** ≤ 30 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_16_pde_structured/01_age_structured_PDE.ipynb)


## What this notebook does

Numerically solves the von Foerster equation for an age-structured population at demographic equilibrium and shows that the analytical stable age distribution $u^*(a) = B \, e^{-\mu a}$ is recovered. This provides the population-structure foundation for the age-structured $SIR_I$ in book §13.2.4. The discretization uses an upwind finite-difference scheme that stably advects the density along the age axis; numerical convergence to the analytical equilibrium is demonstrated.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_16
from shared.verification import assert_within_tolerance
set_seed_chapter_16()
book_style()


## The von Foerster equation

$$\frac{\partial u}{\partial t} + \frac{\partial u}{\partial a} = -\mu(a) u, \qquad u(0, t) = \int_0^{a_{\max}} \beta(a) u(a, t)\, da.$$
With constant mortality $\mu$ and crude birth rate $B = \mu$, the stable age distribution is $u^*(a) = B e^{-\mu a}$, normalized so $\int u^* = 1$.

In [ ]:
from shared.parameters import baseline_chapter_16
p = baseline_chapter_16()
a = np.linspace(0, p['a_max'], p['n_age'])
da = a[1] - a[0]
mu = p['mu_baseline']
B = p['birth_rate']

# Initial condition: deliberately non-equilibrium uniform distribution
u = np.ones_like(a)
u = u / (u.sum() * da)

# Time stepping (upwind finite difference, CFL-stable)
dt = 0.5 * da  # CFL = 0.5 with advection speed 1
T = 400.0
n_steps = int(T / dt)
snapshot_times = [50, 150, 400]
snaps = {}
snap_iter = 0
for step in range(n_steps):
    t_now = step * dt
    # Boundary condition: birth integral = total population times B (with B = mu, replacement)
    total_pop = u.sum() * da
    u_new = u.copy()
    u_new[1:] = u[1:] - dt/da * (u[1:] - u[:-1]) - dt * mu * u[1:]
    u_new[0] = B * total_pop  # newborns = B * population
    u = u_new
    if snap_iter < len(snapshot_times) and t_now >= snapshot_times[snap_iter]:
        snaps[snapshot_times[snap_iter]] = u.copy()
        snap_iter += 1


## Convergence to the analytical equilibrium

In [ ]:
u_eq_analytic = B * np.exp(-mu * a)
u_eq_analytic = u_eq_analytic / (u_eq_analytic.sum() * da)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for tt, snap in snaps.items():
    snap_norm = snap / (snap.sum() * da)
    ax.plot(a, snap_norm, lw=1.5, label=f't = {tt} y')
ax.plot(a, u_eq_analytic, 'k--', lw=2, label='analytic $u^*(a)$')
ax.set_xlabel('age $a$ (years)')
ax.set_ylabel('density $u(a, t)$')
ax.set_title('von Foerster: numerical convergence to stable age distribution')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

u_norm_final = u / (u.sum() * da)
L2_err = np.sqrt(((u_norm_final - u_eq_analytic)**2 * da).sum())
print(f'L2 error vs analytical equilibrium: {L2_err:.4f}')
assert L2_err < 0.05, 'numerical solution should converge to within 5% L2 of analytical eq.'
print('Verified: numerical scheme converges to stable age distribution.')


### Why this matters for epidemic models (Consideration~5)

Linearization fidelity: an age-structured $SIR_I$ that builds on this PDE inherits the *exact same* linearization at the disease-free demographic equilibrium $u^*(a)$. Standard compartmental $\mathcal{R}_0$ calculations are recovered by integrating against $u^*(a)$; the PDE machinery exposes which life-history details (age-specific contact rates, age-specific mortality) the compartmental version is silently averaging over.

## What's next

- Book §13.3 develops the *time-since-infection* density (a similar PDE in a different variable).
- Book §13.4 introduces *spatial* PDEs: reaction-diffusion epidemic models on a 1-D domain.
- Chapter 10's sensitivity machinery applies to PDEs via adjoint methods — beyond this companion's scope but treated computationally in the sensitivity-analysis textbook (year 2 of this sequence).